<table align="left">
    <tr>
        <td style="vertical-align: middle; padding-left: 0px; padding-right: 0px;">
            <a href="https://creativecommons.org/licenses/by/4.0/">
                <img src="https://licensebuttons.net/l/by/4.0/80x15.png" />
            </a>
        </td>
        <td style="vertical-align: middle; padding-left: 5px; padding-right: 0px;">
            <a href="https://opensource.org/licenses/MIT">
                <img src="https://img.shields.io/badge/License-MIT-green.svg" />
            </a>
        </td>
        <td style="vertical-align: middle; padding-left: 15px;">
            &copy; Guillaume Rongier
        </td>
    </tr>
</table>

# Figure 2, A4, & A7

### Imports

In [ ]:
import math
from functools import partial
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.gridspec as gridspec
from matplotlib.ticker import FormatStrFormatter
from matplotlib.lines import Line2D
import matplotlib.patches as mpatches
from matplotlib.patches import ConnectionPatch
import colorsys
import pyvista as pv

import torch
import torch.nn as nn

from voxgan.data.datasets import *
from voxgan.models.base import detach
from voxgan.networks import resnet

### Functions

In [ ]:
def _build_sand_cmap(light_fraction_1,
                     light_fraction_2,
                     light_fraction_3,
                     light_fraction_4,
                     use_gold_sand=False,
                     reverse=False,
                     name='sand'):
    """
    Builds a colormap following a sandy color scheme.

    Parameters
    ----------
    light_fraction_1 : float
        Light fraction of the first color.
    light_fraction_2 : float
        Light fraction of the second color.
    light_fraction_3 : float
        Light fraction of the third color.
    light_fraction_4 : float
        Light fraction of the fourth color.
    use_gold_sand : bool (default False)
        If true, uses Gold Sand as third color, otherwise uses Yuma Sand.
    reverse : bool (default False)
        If true, reverses the color list.
    name : str (default 'sand')
        Name of the colormap.

    Returns
    -------
    cmap
        The colormap.
    """
    mississippi_mud = (15/360, 0.29 + light_fraction_1*0.29, 0.14)
    rio_grande_mud = (22/360, 0.48 + light_fraction_2*0.48, 0.33)
    yuma_sand = (50/360, 0.89 + light_fraction_3*0.89, 0.78)
    drifted_sand = (55/360, 0.94 + light_fraction_4*0.94, 0.40)
    color_list = [colorsys.hls_to_rgb(*mississippi_mud) + (1,),
                  colorsys.hls_to_rgb(*rio_grande_mud) + (1,),
                  colorsys.hls_to_rgb(*yuma_sand) + (1,),
                  colorsys.hls_to_rgb(*drifted_sand) + (1,)]
    if use_gold_sand:
        gold_sand = (46/360, 0.82 + light_fraction_3*0.82, 0.83)
        color_list[2] = colorsys.hls_to_rgb(*gold_sand) + (1,)
    if reverse:
        color_list = color_list[::-1]

    return mcolors.LinearSegmentedColormap.from_list(name, color_list)


sand_light = _build_sand_cmap(0.25, 0.08543330133998818, -0.4679754966612923, 0.0,
                              use_gold_sand=False,
                              name='sand_light')

### Setting

In [ ]:
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = 'Source Sans Pro'
plt.rcParams['font.size'] = 8
plt.rcParams['text.usetex'] = True
plt.rcParams['text.latex.preamble'] = (
    r'\usepackage[T1]{fontenc}'
    r'\usepackage{textgreek}'
    r'\usepackage{sansmath}'
    r'\usepackage{sourcesanspro}'
    r'\sansmath'
)

plt.rcParams['figure.constrained_layout.use'] = False
plt.rcParams['figure.titlesize'] = 9

plt.rcParams['axes.edgecolor'] = '#8C8C8C'
plt.rcParams['axes.labelcolor'] = '#595959'
plt.rcParams['axes.linewidth'] = 0.4
plt.rcParams['axes.labelsize'] = 9
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.titlecolor'] = 'black'
plt.rcParams['axes.titlesize'] = 9
plt.rcParams['xtick.color'] = '#8C8C8C'
plt.rcParams['xtick.labelsize'] = 8
plt.rcParams['xtick.major.pad'] = 3
plt.rcParams['xtick.major.size'] = 3
plt.rcParams['xtick.major.width'] = 0.4
plt.rcParams['xtick.minor.pad'] = 3
plt.rcParams['xtick.minor.size'] = 1.5
plt.rcParams['xtick.minor.width'] = 0.4
plt.rcParams['ytick.color'] = '#8C8C8C'
plt.rcParams['ytick.labelsize'] = 8
plt.rcParams['ytick.major.pad'] = 3
plt.rcParams['ytick.major.size'] = 3
plt.rcParams['ytick.major.width'] = 0.4
plt.rcParams['ytick.minor.pad'] = 3
plt.rcParams['ytick.minor.size'] = 1.5
plt.rcParams['ytick.minor.width'] = 0.4

plt.rcParams['grid.color'] = '#BFBFBF'
plt.rcParams['grid.alpha'] = 0.5
plt.rcParams['grid.linewidth'] = 0.2

plt.rcParams['lines.linewidth'] = 0.6

In [ ]:
palette = ['#003f5c', '#ffa600', '#bc5090']

## 1. MS-SWD data

In [ ]:
model_id = 1

In [ ]:
msswd = pd.read_csv('../outputs/fluvgan_1_training_2_test-msswd_embedding.csv')
msswd = msswd[msswd['Model'] == model_id]
msswd = msswd.reset_index(drop=True)

In [ ]:
distances = np.load('../outputs/fluvgan_1_training_2_test-msswd_distance.npy')
distances = distances[model_id - 1]

## 2. Generated and training samples

In [ ]:
np.random.seed(101010)

idx_sub = np.random.choice(msswd[msswd['Test_sample'].notna()].index, size=4)

if model_id == 1:
    sub_msswd = msswd.loc[idx_sub, ['Dimension_1', 'Dimension_2']].sort_values('Dimension_2', ascending=False)
elif model_id == 2:
    sub_msswd = msswd.loc[idx_sub, ['Dimension_1', 'Dimension_2']].sort_values('Dimension_1')
    # sub_msswd = pd.concat([sub_msswd.iloc[1:2], sub_msswd.iloc[:1], sub_msswd.iloc[2:]])
else:
    sub_msswd = msswd.loc[idx_sub, ['Dimension_1', 'Dimension_2']].sort_values('Dimension_1')
    sub_msswd = pd.concat([sub_msswd.iloc[:1], sub_msswd.iloc[1:].sort_values('Dimension_2', ascending=False)])

sample_idx_swd = []
for i in range(len(sub_msswd)):
    l = int(sub_msswd.iloc[i].name)
    neighbors = distances.shape[0]//2 + np.argsort(distances[:, msswd['Test_sample'].isna()][l])
    sample_idx_swd += [l, int(msswd.loc[neighbors[0]].name)]

In [ ]:
checkpoint_path = '../outputs/fluvgan_1_training_1_architecture_dcgan_4_' + str(model_id) + '_training_checkpoint_iteration_44400.pt'

generator = resnet.DeepGenerator3d(nz=100,
                                   nc=2,
                                   nl=(2, 5, 5),
                                   weight_normalization=nn.utils.parametrizations.spectral_norm,
                                   activation=partial(nn.LeakyReLU, negative_slope=0.2, inplace=True),
                                   use_double_conv=False,
                                   use_double_resblocks=False,
                                   use_attention=False,
                                   split_z=False)

generator = nn.DataParallel(generator)
checkpoint = torch.load(checkpoint_path, torch.device('cpu'))
generator.load_state_dict(checkpoint['generator'])
generator = generator.module
generator.eval();

In [ ]:
training_data_dir_path = '/Volumes/One Touch/Activities/5_Postdoc_CSIRO/Experiments/Fluvial_Deposits/Data/FluvDepoSet'

transform = [
    Compose([Crop(((1, 3), (8, 24), (0, 128), None)), FillNaN((0., 'max+1')), Scale(((0, 1), None)), ToTensor()]),
    Compose([Crop(((1, 3), (12, 28), (72, 200), None)), FillNaN((0., 'max+1')), Scale(((0, 1), None)), ToTensor()]),
]

In [ ]:
samples_swd = []
for idx in sample_idx_swd:
    if np.isnan(msswd.loc[idx, 'Test_sample']) == True:
        z = torch.tensor(msswd.loc[idx][['Latent_' + str(i + 1) for i in range(100)]].to_numpy(),
                         dtype=torch.float)
        z = z.reshape((1,) + z.shape + (1, 1, 1))
        samples_swd.append(0.5*detach(generator(z)) + 0.5)
    else:
        with h5py.File(training_data_dir_path + '/sample_' + str(int(msswd.loc[idx, 'Test_sample'])) + '.h5') as file:
            samples_swd.append(0.5*transform[int(msswd.loc[idx, 'Transform']) - 1]({'data': file['model'][:]})['data'].unsqueeze(0) + 0.5)
samples_swd = torch.concat(samples_swd)

In [ ]:
is_test = False

fig_size = (10, 6)
fig_dpi = 300

window_size = (2*round(int(fig_dpi*fig_size[0])/2), 2*round(int(fig_dpi*fig_size[1])/2))

shape = samples_swd.shape[-3:][::-1]
spacing = (50., 50., 0.5)
grid = pv.ImageData(dimensions=[s + 1 for s in shape], spacing=spacing, origin=(0., 0., 0.))
grid = grid.scale([1., 1., 100.])
grid.cell_data['sample'] = samples_swd[0, 0].flatten()

if is_test == True:
    p = pv.Plotter(border=True, window_size=window_size, lighting=None)
else:
    p = pv.Plotter(off_screen=True, notebook=False, border=False, window_size=window_size, lighting=None)

actor_grid = p.add_mesh(grid, cmap=sand_light, clim=(0., 1.), show_scalar_bar=False,
                        ambient=0., diffuse=1., specular=0.)
bound_mesh = p.add_mesh(pv.Box(grid.bounds).outline(), color='black', line_width=8)

dims = np.asarray(grid.bounds).reshape(-1, 2)
dims = dims[:, 1] - dims[:, 0]
p.camera_position = 'yz'
p.camera.azimuth = 230
p.camera.elevation = 30
p.camera.focal_point = (dims[0]/2., dims[1]/2. - 125., -210.)
p.camera.zoom(1.67)

if is_test == True:
    p.show(jupyter_backend='static')
else:
    samples_swd_images = np.empty(samples_swd.shape[:2] + window_size[::-1] + (4,),
                                  dtype=np.int16)
    for j in range(samples_swd_images.shape[0]):
        for i in range(samples_swd_images.shape[1]):
            _ = p.remove_actor(actor_grid)
            grid['sample'] = samples_swd[j, i].flatten()
            cmap = sand_light if i == 0 else 'viridis'
            actor_grid = p.add_mesh(grid, cmap=cmap, clim=(0., 1.), show_scalar_bar=False,
                                    ambient=0., diffuse=1., specular=0.)
            samples_swd_images[j, i] = p.screenshot(transparent_background=True)

## 4. LoS data

In [ ]:
los = pd.read_csv('../outputs/fluvgan_1_training_3_test-los_metric.csv')
los = los[los['Model'] == model_id]

## 5. Generated samples

In [ ]:
latent_colnames = ['Latent_' + str(i + 1) for i in range(100)]
z = torch.tensor(los.loc[[los['LoS'].idxmin(), np.abs(los['LoS'] - los['LoS'].median()).idxmin()], latent_colnames].to_numpy(),
                 dtype=torch.float)
z = z.reshape(z.shape + (1, 1, 1))

samples_los = 0.5*detach(generator(z)) + 0.5

In [ ]:
is_test = False

fig_size = (10, 6)
fig_dpi = 300

window_size = (2*round(int(fig_dpi*fig_size[0])/2), 2*round(int(fig_dpi*fig_size[1])/2))

shape = samples_los.shape[-3:][::-1]
spacing = (50., 50., 0.5)
grid = pv.ImageData(dimensions=[s + 1 for s in shape], spacing=spacing, origin=(0., 0., 0.))
grid = grid.scale([1., 1., 100.])
grid.cell_data['sample'] = samples_los[0, 0].flatten()

if is_test == True:
    p = pv.Plotter(border=True, window_size=window_size, lighting=None)
else:
    p = pv.Plotter(off_screen=True, notebook=False, border=False, window_size=window_size, lighting=None)

actor_grid = p.add_mesh(grid, cmap=sand_light, clim=(0., 1.), show_scalar_bar=False,
                        ambient=0., diffuse=1., specular=0.)
bound_mesh = p.add_mesh(pv.Box(grid.bounds).outline(), color='black', line_width=8)

p.camera_position = 'yz'
p.camera.azimuth = 230
p.camera.elevation = 30
p.camera.focal_point = (3200., 3075., -210.)
p.camera.zoom(1.67)
p.camera.position = (-6571.398931712063, -8445.099781539067, 9176.646778139224)

if is_test == True:
    p.show(jupyter_backend='static')
else:
    sample_los_images = np.empty(samples_los.shape[:1] + (samples_los.shape[1] + 1,) + window_size[::-1] + (4,),
                                 dtype=np.int16)
    for j in range(sample_los_images.shape[0]):
        for i in range(sample_los_images.shape[1] - 1):
            _ = p.remove_actor(actor_grid)
            grid['sample'] = samples_los[j, i].flatten()
            cmap = sand_light if i == 0 else 'viridis'
            actor_grid = p.add_mesh(grid, cmap=cmap, clim=(0., 1.), show_scalar_bar=False,
                                    ambient=0., diffuse=1., specular=0.)
            sample_los_images[j, i] = p.screenshot(transparent_background=True)

In [ ]:
cmap_los = mcolors.LinearSegmentedColormap.from_list('facies',
                                                     [mcolors.to_rgb('#ca0020'),
                                                      mcolors.to_rgb('#92c5de')])

In [ ]:
is_test = False

fig_size = (10, 6)
fig_dpi = 300

window_size = (2*round(int(fig_dpi*fig_size[0])/2), 2*round(int(fig_dpi*fig_size[1])/2))

sub_samples_los = samples_los[:, 1, 1:] >= samples_los[:, 1, :-1]

shape = sub_samples_los.shape[-3:][::-1]
spacing = (50., 50., 0.5)
grid = pv.ImageData(dimensions=[s + 1 for s in shape], spacing=spacing, origin=(0., 0., 0.))
grid = grid.scale([1., 1., 100.])
grid.cell_data['sample'] = sub_samples_los[0].flatten()

if is_test == True:
    p = pv.Plotter(border=True, window_size=window_size, lighting=None)
else:
    p = pv.Plotter(off_screen=True, notebook=False, border=False, window_size=window_size, lighting=None)

actor_grid = p.add_mesh(grid, cmap=cmap_los, clim=(0., 1.), show_scalar_bar=False,
                        ambient=0., diffuse=1., specular=0.)
bound_mesh = p.add_mesh(pv.Box(grid.bounds).outline(), color='black', line_width=8)

p.camera_position = 'yz'
p.camera.azimuth = 230
p.camera.elevation = 30
p.camera.focal_point = (3200., 3075., -210.)
p.camera.zoom(1.67)
p.camera.position = (-6571.398931712063, -8445.099781539067, 9176.646778139224)

if is_test == True:
    p.show(jupyter_backend='static')
else:
    for j in range(sample_los_images.shape[0]):
        _ = p.remove_actor(actor_grid)
        grid['sample'] = sub_samples_los[j].flatten()
        actor_grid = p.add_mesh(grid, cmap=cmap_los, clim=(0., 1.), show_scalar_bar=False,
                                ambient=0., diffuse=1., specular=0.)
        sample_los_images[j, -1] = p.screenshot(transparent_background=True)

## 6. Final figure

In [ ]:
def plot_los(ax, los, color, axsamples):
    ax.hist(los['LoS'], bins='auto', color=color, ec=color, zorder=-1)
    ax.axvline(0.5, ymax=1.06, c='#ef3b2c', lw=0.6, ls=(0, (8, 4)), clip_on=False)
    ax.axvline(los['LoS'].median(), ymax=1.06, c='#bcbddc', lw=0.6, ls=(0, (8, 4)), clip_on=False)
    ax.grid(True, axis='y', clip_on=False)
    ax.set(xlim=(0.495, 1), yscale='log', xlabel='Fraction of cells honoring the law of superposition ($f_s$)', ylabel='Number of samples')
    ax.xaxis.set_major_formatter(FormatStrFormatter('%g'))
    ax.spines.bottom.set_bounds(0.5, 1)
    ax.spines['left'].set_visible(False)
    ax.set_yticks([1, 10, 100, 1000], labels=['1', '10', '100', r'1\,000'])
    ax.annotate('Samples with\nrandom values', (0.5, 30), (0.545, 12000),
                arrowprops={'arrowstyle': '-|>', 'connectionstyle': 'angle3, angleA=90, angleB=3', 'color': '#ef3b2c', 'lw': 0.4},
                c='#ef3b2c', fontsize=8, va='top')
    fig.add_artist(ConnectionPatch(xyA=(math.trunc(los['LoS'].min()*1000)/1000, 1), coordsA=ax.transData, xyB=(0.5, 0.5), coordsB=axsamples[0].transAxes,
                                   arrowstyle='<|-', connectionstyle='angle3, angleA=-10, angleB=120', color='#fd8d3c', lw=0.4, zorder=0))
    ax.text(0.66, 12000, 'Sample with\nthe lowest fraction', c='#fd8d3c', fontsize=8, ha='left', va='top')
    fig.add_artist(ConnectionPatch(xyA=(math.trunc(los['LoS'].median()*1000)/1000, 30), coordsA=ax.transData, xyB=(0.5, 0.5), coordsB=axsamples[1].transAxes,
                                   arrowstyle='<|-', connectionstyle='angle3, angleA=-25, angleB=90', color='#bcbddc', lw=0.4, zorder=0))
    ax.text(0.905, 12000, 'Sample closest\nto the median', c='#bcbddc', fontsize=8, ha='left', va='top')


def plot_sample(ax, sample, margin=200):
    ax.axis('off')
    ax.imshow(sample, clip_on=False)
    ax.set(xlim=(margin, sample.shape[-2] - margin), ylim=(sample.shape[-3] - margin, margin))


def plot_scale(ax, i):
    if i == 3:
        ax.text(-0.09, 0.58, r'8~m$^*$', fontsize=8, color='#8C8C8C', ha='right', va='center', transform=ax.transAxes)
    else:
        xytext = (-0.1, 1.17) if i == 2 else (-0.15, 0.92)
        rad = '0.8' if i == 2 else '1.2'
        ax.annotate(r'8~m$^*$', (-0.06, 0.63), xytext=xytext, xycoords=ax.transAxes, textcoords=ax.transAxes, fontsize=8, color='#8C8C8C',
                    arrowprops=dict(arrowstyle='-|>', connectionstyle='arc3, rad=' + rad, color='#8C8C8C', lw=0.4), bbox=dict(boxstyle='square, pad=-0.1', fc='none', ec='none'))
    ax.text(0.11, 0.43, r'6\,400~m', fontsize=8, color='#8C8C8C', ha='center', va='top', rotation=-38, transform=ax.transAxes)
    ax.text(0.83, 0.35, r'6\,400~m', fontsize=8, color='#8C8C8C', ha='center', va='top', rotation=30, transform=ax.transAxes)
    ax.text(1.1, 1.2, r'\begin{center}\noindent$^*$Vertical exaggeration:\\[-0.1ex]$\times$ 100\end{center}', fontsize=8, color='#8C8C8C', ha='center', va='baseline', transform=ax.transAxes)


def plot_sample_link(i, ax, c):
    fig.add_artist(ConnectionPatch(xyA=msswd.loc[i, ['Dimension_1', 'Dimension_2']], coordsA=axswd.transData,
                                   xyB=(0.5, 0.5), coordsB=ax.transAxes, color=c, lw=0.4, zorder=-1))


def plot_colorbar(ax, label, cmap='viridis', vmin=0, vmax=0):
    ax.axis('off')
    cax = ax.inset_axes([0.1, 0.1, 0.8, 0.11])
    cbar = fig.colorbar(mpl.cm.ScalarMappable(cmap=cmap), cax=cax, orientation='horizontal', ticks=[])
    cbar.ax.text(-0.05, 0.4, '0', color='#8C8C8C', va='center', ha='right', transform=cbar.ax.transAxes)
    cbar.ax.text(1.06, 0.4, '1', color='#8C8C8C', va='center', ha='left', transform=cbar.ax.transAxes)
    cbar.ax.text(0.5, 1.5, label, color='#595959', fontsize=9, va='bottom', ha='center', transform=cbar.ax.transAxes)


def plot_legend_mds(ax, labels, colors, markers):
    ax.set_axis_off()
    points = [Line2D([0], [0], c=c, marker=m, markersize=4.5, markeredgewidth=0.5, markeredgecolor='white', ls='') for c, m in zip(colors, markers)]
    ax.legend(points, labels, loc='lower center', fontsize=8, labelcolor='#8C8C8C', frameon=False, handletextpad=0.4, bbox_to_anchor=(0, -0.1, 1, 1))


def plot_legend_los(ax, labels, colors):
    ax.set_axis_off()
    patches = [mpatches.Patch(fc=c, ec='#8C8C8C', lw=0.4) for c in colors]
    ax.legend(patches, labels, loc='lower center', fontsize=8, labelcolor='#8C8C8C', frameon=False, handletextpad=0.6, bbox_to_anchor=(0, -0.1, 1, 1))

In [ ]:
fig = plt.figure(figsize=(18/2.54, 19/2.54))
gspec = gridspec.GridSpec(nrows=9, ncols=6, figure=fig, left=0.077, bottom=0.054, right=0.985, top=0.992,
                          height_ratios=(1, 1, 1, 1, 1, 1, 1, 1, 1.5), hspace=0.2)

axswd = plt.subplot(gspec[1:6, :4])
axswdsamples = [plt.subplot(gspec[0, i]) for i in range(4)]
axswdsamples += [plt.subplot(gspec[i, j + 4]) for i in range(6) for j in range(2)]
axsleg = [plt.subplot(gspec[6, :2]), plt.subplot(gspec[6, 2:4]), plt.subplot(gspec[6, 4]), plt.subplot(gspec[6, 5])]
axlos = plt.subplot(gspec[8, :])
axlossamples = [plt.subplot(gspec[7, i]) for i in range(6)]

for i, (idx, c) in enumerate(zip(sample_idx_swd, 4*['#d1d1d1', palette[model_id - 1]])):
    plot_sample_link(idx, axswdsamples[2*i], c)
    plot_sample(axswdsamples[2*i], samples_swd_images[i, 0])
    plot_sample(axswdsamples[2*i + 1], samples_swd_images[i, 1])

axswd.set_facecolor('none')
is_valid = msswd['Test_sample'].notna()
axswd.scatter(msswd.loc[is_valid, 'Dimension_1'], msswd.loc[is_valid, 'Dimension_2'], s=20, c='#d1d1d1', marker='s', lw=0.5, ec='white')
is_valid = msswd['Test_sample'].isna()
axswd.scatter(msswd.loc[is_valid, 'Dimension_1'], msswd.loc[is_valid, 'Dimension_2'], s=20, c=palette[model_id - 1], lw=0.5, ec='white')
axswd.set(aspect='equal', xlabel='Dimension 1', ylabel='Dimension 2')
axswd.spines[['right', 'top']].set_visible(True)
axswd.xaxis.set_major_formatter(FormatStrFormatter('%g'))
axswd.yaxis.set_major_formatter(FormatStrFormatter('%g'))

axlos.set_zorder(-1)
plot_los(axlos, los, palette[model_id - 1], [axlossamples[2], axlossamples[5]])
plot_sample(axlossamples[0], sample_los_images[0, 0])
plot_sample(axlossamples[1], sample_los_images[0, 1])
plot_sample(axlossamples[2], sample_los_images[0, 2])
plot_sample(axlossamples[3], sample_los_images[1, 0])
plot_sample(axlossamples[4], sample_los_images[1, 1])
plot_sample(axlossamples[5], sample_los_images[1, 2])

plot_scale(axswdsamples[14], model_id)

plot_legend_mds(axsleg[0], [r'Sample from model ' + str(model_id) + ' of~~~4~~~(see figure 1)', 'Test sample'],
                colors=[palette[model_id - 1], '#d1d1d1'], markers=['o', 's'])
axsleg[0].scatter(0.692, 0.4, s=64, edgecolors='#8C8C8C', facecolors='white', lw=0.4, transform=axsleg[0].transAxes, clip_on=False)
plot_legend_los(axsleg[1], ['Law of superposition honored', 'Law of superposition violated'], colors=['#92c5de', '#ca0020'])
plot_colorbar(axsleg[2], 'Fraction of\ncoarse sediments', sand_light)
plot_colorbar(axsleg[3], 'Normalized\ndeposition time')

fig.align_ylabels([axswd, axlos]);

In [ ]:
fig_id = '2' if model_id == 1 else 'a4' if model_id == 2 else 'a7'
fig.savefig('./figure_' + fig_id + '.pdf', dpi=400)